In [11]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
block_size = 8
batch_size = 4
max_iters = 1000
# eval_interval = 2500
learning_rate = 3e-4
eval_iters = 250
# dropout = 0.2
    # if we use this, reduces overfitting for neural network

cuda


In [12]:
with open('wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(text))
print(chars)
vocab_size = len(chars)


['\n', ' ', '!', '"', '&', "'", '(', ')', '*', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '\ufeff']


In [13]:
string_to_int = { ch:i for i,ch in enumerate(chars) }
int_to_string = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
# print(data[:100])


In [14]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch('train')
print('inputs:')
# print(x.shape)
print(x)
print('targets:')
print(y)



inputs:
tensor([[62, 67, 60,  1, 73, 68, 60, 58],
        [ 1,  1,  1,  1,  1,  1,  1,  1],
        [76, 58, 57,  1, 61, 62, 66, 11],
        [54, 62, 57,  1, 50, 58, 55,  9]], device='cuda:0')
targets:
tensor([[67, 60,  1, 73, 68, 60, 58, 73],
        [ 1,  1,  1,  1,  1,  1,  1,  1],
        [58, 57,  1, 61, 62, 66, 11,  1],
        [62, 57,  1, 50, 58, 55,  9,  1]], device='cuda:0')


In [15]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [16]:
class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
        
    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index)
        
# B meaning batch
# T meaning time
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    def generate(self, index, max_new_tokens):
        # index is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self.forward(index)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            index_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            index = torch.cat((index, index_next), dim=1) # (B, T+1)
        return index

model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


S(oQ&)
g,y:IsI15IVnKj(HcKJ1-t0Chd.Y&9Y3q7EDeNaa.MIfLsWsIvehzUI*&9-U4nlzM2Vd?5 5)-0ht p-exE7QcjOLs)pVFPZIOjqm VLQIAM!)b1P.A&z?_M"ZN-VnUQluY R7(OXpo;Mxu.lh﻿Ugt_OG﻿]K"8ki7(?PJ[aoAMV1Z0Oe1A﻿DH(_X8XdAku'q,Rrr-V93mBWl)XS8-b1u8 ln&-sb(30L*Vz[4P?,dA.lzCpYYN6MsFqu4kBSr-i3]mgL F5HVnfn&)Jv,ftfvTiL&w[uU2ts)O-sXL)﻿sn3wKtbEszzMo*.9jDU*?D7V"j.3SeprjpY.D.312,H80Sj,)(T3﻿Dnc4"RoX6;]-sxgj:PBJ_a7]_xRPW4rv8.xoJJS]ypau0mD!prA0fLARsz9nVUSpw73VrrwuuELcRsjTu4o_
wfuiW83
Sx_lL7zF3ehMTki(3!4&5CATicI!O02Vvb-s9SOROVwp5pHF4rf


In [17]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# AdamW is a modification to Adam that adds weight decay (prevent it from having super low level performance)

for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

step: 0, train loss: 4.893, val loss: 4.888
step: 250, train loss: 4.819, val loss: 4.824
step: 500, train loss: 4.746, val loss: 4.762
step: 750, train loss: 4.700, val loss: 4.678
4.690408229827881


In [18]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


Oas]z65b]gENJve,km
H8g_csF4X F?ONtv02CZjyQWTYRl:JQc5C9P97Wrd.W6cuA﻿&Y* h*NJ"r)﻿BB7﻿I[hYD"ROfoSz_2m""'3!H;B-i9hg*z[(f(9ma.3TY8N6M[;IlmVCJiLY"Ooqc,lJ0AMy8&8hwA8hFUMVkCGXkBk2TBBBDf!VWACS67LBa(TN,d..OjU*4TY&9D4nR&FX!(Yniq?pBsSQrr(
WvH;20ZxSkKQcetHXPEC81dw,"7wL'u]1dc.0Niyq0fu];pYi!DSbQm87oAxSZGz640Q06CuOVbWRMovF'glMHjt'saaA2DCO(*vXSwuegw)CZMEPemWZN684Hq4rZB*8;I'f:k2'i8C?4Pxv(zhR*eHgJutGRMv:2ZYlHiAb[aXJBJB*ny33wzdY[Q'4bDjqq0RoB*U5&u1O5)﻿tXQz9;EKXrrcs68XPGq)X9e&s"p&6nI3m_Ilk_Om﻿283ma-s_COLA_exOG0I5CAed
